In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import pickle
load_dotenv()

with open("bfl_store.pkl", "rb") as f:
    store = pickle.load(f)


/Users/rahultiwari/Documents/aziz_ai_engineering/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


In [3]:
GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [9]:
chunk_vectors = store['vectors']
chunk_texts = store['text']
chunk_meta = store['meta']
model_name = store['model']


In [27]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(model_name)
def retrieve_similar_chunks(query, top_k=3):
     # Convert the query into an embedding
    query_vector = model.encode([query], convert_to_numpy=True)

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

    # Return (score, index)
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results



def get_context(query, top_k=3):
    results = retrieve_similar_chunks(query, top_k=top_k)
  
    context_parts = []
    for score, idx in results:
        text = chunk_texts[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8920.92it/s]


In [28]:
q_relevant = "What is the foreclosure charge on gold loans?"
results = get_context(q_relevant,top_k=5)

['months, the charge further reduces to 1 percent of the outstanding principal. After 12 months, there is\nno foreclosure charge — the customer pays only the outstanding principal plus interest accrued to date.\nFor all closures, a branch visit is required and the NOC is issued within 30 minutes.\n7.3 Gold Auction Policy', 'applicable charges.\nIf the loan is closed within the first three months, a foreclosure charge of 2 percent on the outstanding\nprincipal plus all accrued interest applies. A branch visit is mandatory for this closure, and the customer\nmust present the original pledge receipt.', 'vaults that are fully insured and monitored by 24-hour CCTV surveillance.\n7.2 Gold Loan Foreclosure Policy\nGold loan foreclosure refers to early closure of the loan before the completion of the agreed tenure. This\nis the most frequently raised query from gold loan customers, so agents must be completely clear on the\napplicable charges.', 'from 3 months to 24 months, with customers havi

In [30]:
print(results)

months, the charge further reduces to 1 percent of the outstanding principal. After 12 months, there is
no foreclosure charge — the customer pays only the outstanding principal plus interest accrued to date.
For all closures, a branch visit is required and the NOC is issued within 30 minutes.
7.3 Gold Auction Policy

applicable charges.
If the loan is closed within the first three months, a foreclosure charge of 2 percent on the outstanding
principal plus all accrued interest applies. A branch visit is mandatory for this closure, and the customer
must present the original pledge receipt.

vaults that are fully insured and monitored by 24-hour CCTV surveillance.
7.2 Gold Loan Foreclosure Policy
Gold loan foreclosure refers to early closure of the loan before the completion of the agreed tenure. This
is the most frequently raised query from gold loan customers, so agents must be completely clear on the
applicable charges.

from 3 months to 24 months, with customers having the option of b

In [32]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = GOOD_PROMPT | llm | StrOutputParser()

In [37]:
answer = chain.invoke({
    "context":results,
    "question": "tell me about the context"
})

In [38]:
print(answer)

The context outlines the foreclosure policy for gold loans at Bajaj Finance, detailing charges for early closure based on the time elapsed since loan disbursement. A foreclosure charge of 2 percent applies if the loan is closed within the first three months, reducing to 1 percent from 3 to 12 months, after which there are no charges. Additionally, it describes the gold auction policy for loans that are not repaid or renewed within 30 days of a formal notice.


In [41]:
query = "what is interest on personal loans?"

context = get_context(query, top_k=3)

answer = chain.invoke({
    "context": context,
    "question": query
})

answer

['the BajajBot AI assistant knowledge base. For structured lookup tables, refer to the companion Policy\nReference Document v4.0.\nSection 1 — Personal Loan: Eligibility, Rates & Charges\n1.1 Who Can Apply for a Bajaj Finance Personal Loan\nBajaj Finance personal loans are designed for both salaried employees and self-employed', 'employment category (salaried or self-employed), and the loan amount requested. All rates quoted\nbelow are effective from April 2025.\nSalaried applicants with a CIBIL score of 750 or above receive the most favourable interest rates,\nranging from 11 percent to 13 percent per annum. These applicants may borrow up to Rs 40 lakhs for a', 'from 3 months to 24 months, with customers having the option of bullet repayment (principal and\ninterest paid together at maturity) or monthly interest payments with principal at maturity.\nInterest rates for gold loans range from 9.5 to 18 percent per annum. High-value loans above Rs 5 lakhs']


'Interest rates for Bajaj Finance personal loans range from 11 percent to 13 percent per annum for salaried applicants with a CIBIL score of 750 or above. For gold loans, the interest rates range from 9.5 to 18 percent per annum.'

In [42]:
while True:
    query = input("Enter your question (or 'exit' to quit): ")
    if query.lower() == 'exit':
        break

    context = get_context(query, top_k=3)

    answer = chain.invoke({
        "context": context,
        "question": query
    })

    print(answer)

['loan application.\nSalaried applicants must have a minimum of one year of continuous service with their current employer\nand at least two years of total work experience. Self-employed professionals must have been practising\nin the same profession for a minimum of three years. Doctors, Chartered Accountants, and Architects', 'the BajajBot AI assistant knowledge base. For structured lookup tables, refer to the companion Policy\nReference Document v4.0.\nSection 1 — Personal Loan: Eligibility, Rates & Charges\n1.1 Who Can Apply for a Bajaj Finance Personal Loan\nBajaj Finance personal loans are designed for both salaried employees and self-employed', 'employment category (salaried or self-employed), and the loan amount requested. All rates quoted\nbelow are effective from April 2025.\nSalaried applicants with a CIBIL score of 750 or above receive the most favourable interest rates,\nranging from 11 percent to 13 percent per annum. These applicants may borrow up to Rs 40 lakhs for a']


## Testing code 

In [23]:
print(chunk_texts[104])
print(chunk_texts[102])
print(chunk_texts[101])
print(chunk_texts[99])
print(chunk_texts[105])

months, the charge further reduces to 1 percent of the outstanding principal. After 12 months, there is
no foreclosure charge — the customer pays only the outstanding principal plus interest accrued to date.
For all closures, a branch visit is required and the NOC is issued within 30 minutes.
7.3 Gold Auction Policy
applicable charges.
If the loan is closed within the first three months, a foreclosure charge of 2 percent on the outstanding
principal plus all accrued interest applies. A branch visit is mandatory for this closure, and the customer
must present the original pledge receipt.
vaults that are fully insured and monitored by 24-hour CCTV surveillance.
7.2 Gold Loan Foreclosure Policy
Gold loan foreclosure refers to early closure of the loan before the completion of the agreed tenure. This
is the most frequently raised query from gold loan customers, so agents must be completely clear on the
applicable charges.
from 3 months to 24 months, with customers having the option of bull

In [22]:
results

[(0.6952211260795593, 104),
 (0.6354900002479553, 102),
 (0.6307044625282288, 101),
 (0.5885403156280518, 99),
 (0.5172236561775208, 105)]

In [24]:
q_irrelevant = "How do I bake a chocolate cake?"
results_irrelevant = retrieve_similar_chunks(q_irrelevant, top_k=5) 

In [25]:
results_irrelevant

[(0.13226991891860962, 4),
 (0.12299858778715134, 37),
 (0.10525134205818176, 30),
 (0.0954020768404007, 29),
 (0.09211493283510208, 11)]